In [3]:
import zipfile
import json
import joblib
import time
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

# 1. Đọc dữ liệu từ dataset.zip
zip_path = "../data/dataset.zip"
with zipfile.ZipFile(zip_path) as z:
    csv_filename = [f for f in z.namelist() if f.endswith(".csv")][0]
    with z.open(csv_filename) as f:
        df = pd.read_csv(f)

X = df.drop(columns=["Potability"])
y = df["Potability"]

# 2. Phân chia dữ liệu (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 3. Tạo Pipeline (Chống rò rỉ dữ liệu)
knn_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('knn', KNeighborsClassifier())
])

rf_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('rf', RandomForestClassifier(random_state=42))
])

# 4. Tinh chỉnh siêu tham số + Đo thời gian
# --- Mô hình 1: KNN ---
print("--- 1. Huấn luyện KNN ---")
param_grid_knn = {'knn__n_neighbors': [3, 5, 7, 9, 11], 'knn__weights': ['uniform', 'distance']}
grid_knn = GridSearchCV(knn_pipeline, param_grid_knn, cv=5, scoring='accuracy', n_jobs=-1)

start_train_knn = time.time()
grid_knn.fit(X_train, y_train)
knn_train_time = time.time() - start_train_knn

start_pred_knn = time.time()
best_knn = grid_knn.best_estimator_
pred_knn = best_knn.predict(X_test)
knn_infer_time = time.time() - start_pred_knn

# --- Mô hình 2: Random Forest ---
print("--- 2. Huấn luyện Random Forest ---")
param_grid_rf = {'rf__n_estimators': [50, 100, 200], 'rf__max_depth': [None, 10, 20]}
grid_rf = GridSearchCV(rf_pipeline, param_grid_rf, cv=5, scoring='accuracy', n_jobs=-1)

start_train_rf = time.time()
grid_rf.fit(X_train, y_train)
rf_train_time = time.time() - start_train_rf

start_pred_rf = time.time()
best_rf = grid_rf.best_estimator_
pred_rf = best_rf.predict(X_test)
rf_infer_time = time.time() - start_pred_rf

# 5. Lưu CẢ HAI MÔ HÌNH cho Backend / Frontend sử dụng
joblib.dump(best_knn, "../models/knn_model.joblib")
joblib.dump(best_rf, "../models/rf_model.joblib")
print("✅ Đã lưu models/knn_model.joblib và models/rf_model.joblib")

# Lưu mô hình thắng cuộc mặc định vào model.joblib
acc_knn = accuracy_score(y_test, pred_knn)
acc_rf = accuracy_score(y_test, pred_rf)

if acc_rf >= acc_knn:
    best_model = best_rf
    best_model_name = "RandomForestClassifier"
    best_acc, best_preds = acc_rf, pred_rf
else:
    best_model = best_knn
    best_model_name = "KNeighborsClassifier"
    best_acc, best_preds = acc_knn, pred_knn

joblib.dump(best_model, "../models/model.joblib")

# 6. Lưu Schema và Metadata
feature_names = list(X.columns)
schema = {
    "features": [{"name": col, "type": str(X[col].dtype), "required": True} for col in feature_names],
    "target": {"name": "Potability", "type": "int64", "classes": [0, 1]}
}
with open("../models/schema.json", "w", encoding="utf-8") as f:
    json.dump(schema, f, indent=4)

metadata = {
    "model_name": best_model_name,
    "accuracy": round(float(best_acc), 4),
    "precision": round(float(precision_score(y_test, best_preds)), 4),
    "recall": round(float(recall_score(y_test, best_preds)), 4),
    "f1_score": round(float(f1_score(y_test, best_preds)), 4),
    "num_features": len(feature_names),
    "features_list": feature_names
}
with open("../models/metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=4)

print("✅ Đã xuất xong Artifacts (model.joblib, schema.json, metadata.json)!")

--- 1. Huấn luyện KNN ---
--- 2. Huấn luyện Random Forest ---
✅ Đã lưu models/knn_model.joblib và models/rf_model.joblib
✅ Đã xuất xong Artifacts (model.joblib, schema.json, metadata.json)!
